## Data Cleaning

Loading the raw file and fixing all the issues we found during extraction.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel('../data/raw/Online Retail.xlsx')
print('raw shape:', df.shape)

In [ ]:
# remove duplicates first
df = df.drop_duplicates()
print('after removing duplicates:', df.shape)

In [ ]:
# lots of missing CustomerIDs - around 135k rows
# we dont want to drop all of them since they still have valid product and price info
# tagging them as Guest instead

df['CustomerID'] = df['CustomerID'].fillna(0).astype(int).astype(str)
df['CustomerID'] = df['CustomerID'].replace('0', 'Guest')
print('Guest rows:', (df['CustomerID'] == 'Guest').sum())

In [ ]:
# missing descriptions - filling with Unknown
df['Description'] = df['Description'].fillna('Unknown').astype(str).str.strip().str.upper()
print('any nulls left in Description:', df['Description'].isnull().sum())

In [ ]:
# removing system/internal rows that are not actual products
junk = ['POSTAGE', 'DOTCOM POSTAGE', 'CRUK COMMISSION', 'MANUAL', 'BANK CHARGES', 'AMAZON FEE']
before = len(df)
df = df[~df['Description'].isin(junk)]
print('junk rows removed:', before - len(df))

In [ ]:
# negative quantities that are NOT cancellations = data errors, dropping those
# keeping actual cancellations (InvoiceNo starts with C) - will flag them
is_cancel = df['InvoiceNo'].astype(str).str.startswith('C')
df = df[~((df['Quantity'] < 0) & (~is_cancel))]

# also dropping zero/negative price rows
df = df[df['UnitPrice'] > 0]
print('shape after removing bad rows:', df.shape)

In [ ]:
# flag cancellations instead of dropping
df['IsCancelled'] = df['InvoiceNo'].astype(str).str.startswith('C')
print('cancelled rows:', df['IsCancelled'].sum())
print('active rows:', (~df['IsCancelled']).sum())

In [ ]:
# feature engineering
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df['TotalRevenue'] = df['Quantity'] * df['UnitPrice']
df['InvoiceYearMonth'] = df['InvoiceDate'].dt.strftime('%Y-%m')
df['MonthName'] = df['InvoiceDate'].dt.strftime('%B')
df['MonthNumber'] = df['InvoiceDate'].dt.month
df['Year'] = df['InvoiceDate'].dt.year
df['Quarter'] = 'Q' + df['InvoiceDate'].dt.quarter.astype(str)
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['Hour'] = df['InvoiceDate'].dt.hour

def get_time_of_day(h):
    if 5 <= h < 12:
        return 'Morning'
    elif 12 <= h < 17:
        return 'Afternoon'
    elif 17 <= h < 21:
        return 'Evening'
    else:
        return 'Night'

df['TimeOfDay'] = df['Hour'].apply(get_time_of_day)
df['IsUK'] = df['Country'].apply(lambda x: 'UK' if x == 'United Kingdom' else 'International')

print('new columns added')
df.head()

In [ ]:
# basket level stuff
basket = df.groupby('InvoiceNo').agg(
    BasketTotal=('TotalRevenue', 'sum'),
    ItemsPerOrder=('Quantity', 'sum')
).reset_index()

def label_basket(v):
    if v < 50:
        return 'Small'
    elif v <= 200:
        return 'Medium'
    else:
        return 'Large'

basket['BasketSize'] = basket['BasketTotal'].apply(label_basket)
df = df.merge(basket[['InvoiceNo', 'BasketSize', 'ItemsPerOrder']], on='InvoiceNo', how='left')

In [ ]:
# customer level - repeat vs one-time
cust = df[df['CustomerID'] != 'Guest'].groupby('CustomerID').agg(
    OrderCount=('InvoiceNo', 'nunique'),
    TotalSpend=('TotalRevenue', 'sum')
).reset_index()

cust['AvgOrderValue'] = (cust['TotalSpend'] / cust['OrderCount']).round(2)
cust['IsRepeatCustomer'] = cust['OrderCount'].apply(lambda x: 'Repeat' if x > 1 else 'One-Time')

df = df.merge(cust[['CustomerID', 'AvgOrderValue', 'IsRepeatCustomer']], on='CustomerID', how='left')
df['IsRepeatCustomer'] = df['IsRepeatCustomer'].fillna('Guest')
df['AvgOrderValue'] = df['AvgOrderValue'].fillna(0)

In [ ]:
# RFM segmentation for actual customers
rfm_df = df[(df['CustomerID'] != 'Guest') & (df['IsCancelled'] == False)].copy()
snapshot = rfm_df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = rfm_df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalRevenue', 'sum')
).reset_index()

rfm['R'] = pd.qcut(rfm['Recency'], 4, labels=[4,3,2,1]).astype(int)
rfm['F'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1,2,3,4]).astype(int)
rfm['M'] = pd.qcut(rfm['Monetary'].rank(method='first'), 4, labels=[1,2,3,4]).astype(int)
rfm['RFM_Score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)

def segment(row):
    r, f, m = row['R'], row['F'], row['M']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champion'
    elif r >= 3 and f >= 3:
        return 'Loyal'
    elif r >= 3 and f <= 2:
        return 'Potential Loyalist'
    elif r == 2 and f >= 2:
        return 'At Risk'
    elif r <= 1 and f >= 3:
        return 'Cant Lose Them'
    elif r <= 1 and f <= 1:
        return 'Lost'
    else:
        return 'Needs Attention'

rfm['CustomerSegment'] = rfm.apply(segment, axis=1)
df = df.merge(rfm[['CustomerID','RFM_Score','CustomerSegment','Recency','Frequency','Monetary']], on='CustomerID', how='left')
df['CustomerSegment'] = df['CustomerSegment'].fillna('Guest')
df['RFM_Score'] = df['RFM_Score'].fillna('N/A')

print('segments:', df['CustomerSegment'].value_counts())

In [ ]:
# save cleaned file
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/cleaned_retail_data.csv', index=False)

print('done!')
print('final shape:', df.shape)